In [ ]:
!pip install oracle-ads[data]

In [1]:
import ads
from ads.dataset.factory import DatasetFactory

/opt/conda/lib/python3.11/site-packages/cerberus/__init__.py:13: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


In [2]:
ads.set_auth(auth="resource_principal")

In [3]:
bucket_name = "ds-lab-bucket-01"
namespace = "ax9nj9bzjzwo"

employees = DatasetFactory.open(
    "oci://ds-lab-bucket-01@ax9nj9bzjzwo/datasets/orcl_attrition.csvorcl_attrition.csv",
    target="Attrition"
)

loop1:   0%|          | 0/4 [00:00<?, ?it/s]

In [4]:
employees.head()

,Age,Attrition,TravelForWork,SalaryLevel,JobFunction,CommuteLength,EducationalLevel,EducationField,Directs,EmployeeNumber,...,WeeklyWorkedHours,StockOptionLevel,YearsinIndustry,TrainingTimesLastYear,WorkLifeBalance,YearsOnJob,YearsAtCurrentLevel,YearsSinceLastPromotion,YearsWithCurrManager,name
0,42,Yes,infrequent,5054,Product Management,2,L2,Life Sciences,1,1,...,80,0,8,0,1,6,4,0,5,Tracy Moore
1,50,No,often,1278,Software Developer,9,L1,Life Sciences,1,2,...,80,1,10,3,3,10,7,1,7,Andrew Hoover
2,38,Yes,infrequent,6296,Software Developer,3,L2,Other,1,4,...,80,0,7,3,3,0,0,0,0,Julie Bell
3,34,No,often,6384,Software Developer,4,L4,Life Sciences,1,5,...,80,0,8,3,3,8,7,3,0,Thomas Adams
4,28,No,infrequent,2710,Software Developer,3,L1,Medical,1,7,...,80,1,6,3,3,2,2,2,2,Johnathan Burnett


In [5]:
employees.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 36 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1470 non-null   int64 
 1   Attrition                 1470 non-null   object
 2   TravelForWork             1470 non-null   object
 3   SalaryLevel               1470 non-null   int64 
 4   JobFunction               1470 non-null   object
 5   CommuteLength             1470 non-null   int64 
 6   EducationalLevel          1470 non-null   object
 7   EducationField            1470 non-null   object
 8   Directs                   1470 non-null   int64 
 9   EmployeeNumber            1470 non-null   int64 
 10  EnvironmentSatisfaction   1470 non-null   int64 
 11  Gender                    1470 non-null   object
 12  HourlyRate                1470 non-null   int64 
 13  JobInvolvement            1470 non-null   int64 
 14  JobLevel                

In [6]:
employees.suggest_recommendations()

Code
Message                                                Variables           Suggested                 Action                                                                                                               
Constant Column                                        Directs             Drop                      Drop                                                                                       .drop_columns(["Directs"])
                                                       Over18              Drop                      Drop                                                                                        .drop_columns(["Over18"])
                                                       WeeklyWorkedHours   Drop                      Drop                                                                             .drop_columns(["WeeklyWorkedHours"])
Contains mostly unique values(100.00%)                 EmployeeNumber      Drop                      Drop                                                                                .drop_columns(["EmployeeNumber"])
                                                                                                     Do nothing                                                                                                           
Strongly correlated with YearsinIndustry(78.22%.)      JobLevel            Drop JobLevel             Drop JobLevel                                                                             .drop_columns(["JobLevel"])
                                                                                                     Drop YearsinIndustry                                                               .drop_columns(["YearsinIndustry"])
                                                                                                     Do nothing                                                                                                           
Strongly correlated with YearsinIndustry(77.29%.)      MonthlyIncome       Drop MonthlyIncome        Drop MonthlyIncome                                                                   .drop_columns(["MonthlyIncome"])
                                                                                                     Drop YearsinIndustry                                                               .drop_columns(["YearsinIndustry"])
                                                                                                     Do nothing                                                                                                           
Strongly correlated with YearsWithCurrManager(76.92%.) YearsOnJob          Drop YearsOnJob           Drop YearsOnJob                                                                         .drop_columns(["YearsOnJob"])
                                                                                                     Drop YearsWithCurrManager                                                     .drop_columns(["YearsWithCurrManager"])
                                                                                                     Do nothing                                                                                                           
Strongly correlated with YearsWithCurrManager(71.44%.) YearsAtCurrentLevel Drop YearsWithCurrManager Drop YearsAtCurrentLevel                                                       .drop_columns(["YearsAtCurrentLevel"])
                                                                                                     Drop YearsWithCurrManager                                                     .drop_columns(["YearsWithCurrManager"])
                                                                                                     Do nothing                                                                                                           
Set Positive Class                                   

In [7]:
transformed_ds = employees.drop_columns(["Directs", "Over18", "WeeklyWorkedHours"])

In [8]:
transformed_ds.shape

(1470, 33)

In [9]:
from ads.dataset.label_encoder import LabelEncoder

In [10]:
transformed_ds.columns

Index(['Age', 'Attrition', 'TravelForWork', 'SalaryLevel', 'JobFunction',
       'CommuteLength', 'EducationalLevel', 'EducationField', 'EmployeeNumber',
       'EnvironmentSatisfaction', 'Gender', 'HourlyRate', 'JobInvolvement',
       'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus',
       'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'OverTime',
       'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction',
       'StockOptionLevel', 'YearsinIndustry', 'TrainingTimesLastYear',
       'WorkLifeBalance', 'YearsOnJob', 'YearsAtCurrentLevel',
       'YearsSinceLastPromotion', 'YearsWithCurrManager', 'name'],
      dtype='object')

In [11]:
transformed_ds['JobFunction'].value_counts()

JobFunction
Software Developer    961
Product Management    446
Admin                  63
Name: count, dtype: int64

In [12]:
from ads.dataset.label_encoder import DataFrameLabelEncoder

In [13]:
ds_encoded = DataFrameLabelEncoder().fit_transform(
    transformed_ds.to_pandas()
)

In [14]:
ds_encoded['JobFunction'].value_counts()

JobFunction
2    961
1    446
0     63
Name: count, dtype: int64

In [15]:
from ads.dataset.helper import upsample

ERROR - Exception
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 3701, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_22507/4194811739.py", line 1, in <module>
    from ads.dataset.helper import upsample
ImportError: cannot import name 'upsample' from 'ads.dataset.helper' (/opt/conda/lib/python3.11/site-packages/ads/dataset/helper.py)
ImportError: cannot import name 'upsample' from 'ads.dataset.helper' (/opt/conda/lib/python3.11/site-packages/ads/dataset/helper.py)

In [16]:
balanced_ds = transformed_ds.up_sample(
    target="Attrition",
    sampler="default"
)

ERROR - Exception
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 3701, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_22507/3010943587.py", line 1, in <module>
    balanced_ds = transformed_ds.up_sample(
                  ^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: ClassificationDataset.up_sample() got an unexpected keyword argument 'target'
TypeError: ClassificationDataset.up_sample() got an unexpected keyword argument 'target'

In [17]:
balanced_ds = transformed_ds.up_sample(sampler="default")

ERROR - Exception
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 3701, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_22507/4179743018.py", line 1, in <module>
    balanced_ds = transformed_ds.up_sample(sampler="default")
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/ads/dataset/classification_dataset.py", line 207, in up_sample
    helper.up_sample(
  File "/opt/conda/lib/python3.11/site-packages/ads/dataset/helper.py", line 659, in up_sample
    raise ModuleNotFoundError(
ModuleNotFoundError: 
                Required package for up-sampling `imblearn` not found.
                Install `imblearn` with `pip install imbalanced-learn`
                and rerun to enable up-sampling.
            
ModuleNotFoundError: 
                Required package for up-sampling `imblearn` not found.
                Inst

In [18]:
!pip install imbalanced-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [imbalanced-learn]


In [1]:
import ads
from ads.dataset.factory import DatasetFactory

/opt/conda/lib/python3.11/site-packages/cerberus/__init__.py:13: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


In [2]:
ads.set_auth(auth="resource_principal")

In [3]:
employees = DatasetFactory.open(
    "oci://ds-lab-bucket-01@ax9nj9bzjzwo/datasets/orcl_attrition.csvorcl_attrition.csv",
    target="Attrition"
)

loop1:   0%|          | 0/4 [00:00<?, ?it/s]

In [4]:
transformed_ds = employees.drop_columns(["Directs", "Over18", "WeeklyWorkedHours"])

In [5]:
balanced_ds = transformed_ds.up_sample(sampler="default")

In [6]:
balanced_ds["Attrition"].value_counts()

Attrition
Yes    1233
No     1233
Name: count, dtype: int64

In [7]:
balanced_ds.suggest_recommendations()

Code
Message                                                Variables           Suggested                 Action                                                                    
Strongly correlated with YearsinIndustry(78.54%.)      JobLevel            Drop YearsinIndustry      Drop JobLevel                                  .drop_columns(["JobLevel"])
                                                                                                     Drop YearsinIndustry                    .drop_columns(["YearsinIndustry"])
                                                                                                     Do nothing                                                                
Strongly correlated with YearsinIndustry(77.72%.)      MonthlyIncome       Drop MonthlyIncome        Drop MonthlyIncome                        .drop_columns(["MonthlyIncome"])
                                                                                                     Drop YearsinIndustry                    .drop_columns(["YearsinIndustry"])
                                                                                                     Do nothing                                                                
Strongly correlated with YearsWithCurrManager(77.85%.) YearsOnJob          Drop YearsOnJob           Drop YearsOnJob                              .drop_columns(["YearsOnJob"])
                                                                                                     Drop YearsWithCurrManager          .drop_columns(["YearsWithCurrManager"])
                                                                                                     Do nothing                                                                
Strongly correlated with YearsWithCurrManager(74.58%.) YearsAtCurrentLevel Drop YearsWithCurrManager Drop YearsAtCurrentLevel            .drop_columns(["YearsAtCurrentLevel"])
                                                                                                     Drop YearsWithCurrManager          .drop_columns(["YearsWithCurrManager"])
                                                                                                     Do nothing                                                                
Set Positive Class                                     Attrition           Yes                       No                          .set_positive_class("No", missing_value=False)
                                                                                                     Yes                        .set_positive_class("Yes", missing_value=False)
                                                                                                     Do nothing

In [8]:
train, validation, test = balanced_ds.train_validation_test_split(
    test_size=0.20,
    validation_size=0.20
)

In [9]:
len(train), len(validation), len(test)

ERROR - Exception
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 3701, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_28502/3594253679.py", line 1, in <module>
    len(train), len(validation), len(test)
    ^^^^^^^^^^
TypeError: object of type 'ADSData' has no len()
TypeError: object of type 'ADSData' has no len()

In [10]:
train

Train Data
Shape of X:(1577, 32)
Shape of y:(1577,)

In [11]:
validation.X.shape, validation.y.shape

((395, 32), (395,))

In [12]:
test.X.shape, test.y.shape

((494, 32), (494,))

In [13]:
validation.X.shape, validation.y.shape

((395, 32), (395,))

In [14]:
test.X.shape, test.y.shape

((494, 32), (494,))

In [15]:
train

Train Data
Shape of X:(1577, 32)
Shape of y:(1577,)

In [16]:
from ads.automl.provider import OracleAutoMLProvider
ml_engine = OracleAutoMLProvider()

ERROR - Exception
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 3701, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_28502/4151155800.py", line 2, in <module>
    ml_engine = OracleAutoMLProvider()
                ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/ads/common/decorator/deprecate.py", line 62, in wrapper
    raise NotSupportedError(msg)
ads.common.decorator.deprecate.NotSupportedError: Class OracleAutoMLProvider is deprecated in 2.6.7 and will be removed in a future release. Oracle AutoML is recommended to be directly instantiated by importing automlx package
NotSupportedError: Class OracleAutoMLProvider is deprecated in 2.6.7 and will be removed in a future release. Oracle AutoML is recommended to be directly instantiated by importing automlx package

In [17]:
import automlx

ERROR - Exception
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 3701, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_28502/3033343066.py", line 1, in <module>
    import automlx
ModuleNotFoundError: No module named 'automlx'
ModuleNotFoundError: No module named 'automlx'

In [18]:
!pip install automlx

ERROR: Could not find a version that satisfies the requirement automlx (from versions: none)
ERROR: No matching distribution found for automlx


In [19]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [20]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

In [21]:
model.fit(train.X, train.y)

ERROR - Exception
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 3701, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_28502/3504535969.py", line 1, in <module>
    model.fit(train.X, train.y)
  File "/opt/conda/lib/python3.11/site-packages/sklearn/base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/sklearn/ensemble/_forest.py", line 359, in fit
    X, y = validate_data(
           ^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/sklearn/utils/validation.py", line 2919, in validate_data
    X, y = check_X_y(X, y, **check_params)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/sklearn/utils/validation.py", line 1314, in check_X_y
    X = check_array(
        ^^^^^^^^^^^^
  File "/opt/conda/

In [22]:
balanced_ds.X.select_dtypes(include=["object"]).columns

ERROR - Exception
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 3701, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_28502/4167068406.py", line 1, in <module>
    balanced_ds.X.select_dtypes(include=["object"]).columns
    ^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/ads/dataset/dataset.py", line 1166, in __getattr__
    attr = getattr(self.df, item)
           ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/pandas/core/generic.py", line 6321, in __getattr__
    return object.__getattribute__(self, name)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'DataFrame' object has no attribute 'X'
AttributeError: 'DataFrame' object has no attribute 'X'

In [23]:
balanced_ds.select_dtypes(include=["object"]).columns

Index(['TravelForWork', 'JobFunction', 'EducationalLevel', 'EducationField',
       'Gender', 'JobRole', 'MaritalStatus', 'OverTime', 'name', 'Attrition'],
      dtype='object')

In [26]:
balanced_encoded = pd.get_dummies(balanced_ds, drop_first=True)

ERROR - Exception
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 3701, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_28502/2200268748.py", line 1, in <module>
    balanced_encoded = pd.get_dummies(balanced_ds, drop_first=True)
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/pandas/core/reshape/encoding.py", line 227, in get_dummies
    result = _get_dummies_1d(
             ^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/pandas/core/reshape/encoding.py", line 251, in _get_dummies_1d
    codes, levels = factorize_from_iterable(Series(data, copy=False))
                                            ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/pandas/core/series.py", line 587, in __init__
    data = sanitize_array(data, index, dtype, copy)
           ^^^^^^^^^^^^

In [25]:
import pandas as pd

In [27]:
balanced_encoded = pd.get_dummies(balanced_ds, drop_first=True)

ERROR - Exception
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/site-packages/IPython/core/interactiveshell.py", line 3701, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_28502/2200268748.py", line 1, in <module>
    balanced_encoded = pd.get_dummies(balanced_ds, drop_first=True)
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/pandas/core/reshape/encoding.py", line 227, in get_dummies
    result = _get_dummies_1d(
             ^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/pandas/core/reshape/encoding.py", line 251, in _get_dummies_1d
    codes, levels = factorize_from_iterable(Series(data, copy=False))
                                            ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/pandas/core/series.py", line 587, in __init__
    data = sanitize_array(data, index, dtype, copy)
           ^^^^^^^^^^^^

In [28]:
balanced_df = balanced_ds.to_pandas()

In [29]:
balanced_encoded = pd.get_dummies(balanced_df, drop_first=True)

In [30]:
X = balanced_encoded.drop("Attrition_Yes", axis=1)
y = balanced_encoded["Attrition_Yes"]

In [31]:
from sklearn.model_selection import train_test_split

In [32]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)

X_validation, X_test, y_validation, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

In [33]:
model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [34]:
y_pred = model.predict(X_validation)

In [35]:
from sklearn.metrics import accuracy_score

In [36]:
accuracy_score(y_validation, y_pred)

0.9634888438133874

In [37]:
from sklearn.metrics import confusion_matrix

In [38]:
confusion_matrix(y_validation, y_pred)

array([[234,  13],
       [  5, 241]])

In [39]:
from sklearn.metrics import classification_report

In [40]:
print(classification_report(y_validation, y_pred))

              precision    recall  f1-score   support

       False       0.98      0.95      0.96       247
        True       0.95      0.98      0.96       246

    accuracy                           0.96       493
   macro avg       0.96      0.96      0.96       493
weighted avg       0.96      0.96      0.96       493



In [41]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight={0:1, 1:2}
)

model.fit(X_train, y_train)

y_pred = model.predict(X_validation)

from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(y_validation, y_pred))
print(classification_report(y_validation, y_pred))

[[237  10]
 [  4 242]]
              precision    recall  f1-score   support

       False       0.98      0.96      0.97       247
        True       0.96      0.98      0.97       246

    accuracy                           0.97       493
   macro avg       0.97      0.97      0.97       493
weighted avg       0.97      0.97      0.97       493



In [42]:
y_prob = model.predict_proba(X_validation)

In [43]:
y_prob[:5]

array([[0.77, 0.23],
       [0.82, 0.18],
       [0.01, 0.99],
       [0.26, 0.74],
       [0.  , 1.  ]])

In [44]:
y_prob_yes = y_prob[:, 1]

In [45]:
threshold = 0.4

y_pred_new = (y_prob_yes >= threshold).astype(int)

In [46]:
from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(y_validation, y_pred_new))
print(classification_report(y_validation, y_pred_new))

[[211  36]
 [  4 242]]
              precision    recall  f1-score   support

       False       0.98      0.85      0.91       247
        True       0.87      0.98      0.92       246

    accuracy                           0.92       493
   macro avg       0.93      0.92      0.92       493
weighted avg       0.93      0.92      0.92       493

